# 22. Unsupervised clustering: K-means and DBSCAN

**Level:** Intermediate

**Prerequisites:** 07 and 11: arrays and learning settings.

[Course index](00_START_HERE.ipynb) | [Previous](21_Bias_Variance_Overfitting_and_Tuning.ipynb) | [Next](23_PCA_and_Dimensionality_Reduction.ipynb)

**Topics:** Unsupervised learning, K-means, DBSCAN, Clustering, Silhouette, Adjusted Rand index.

**Matching script:** `examples\22_KMeans_and_DBSCAN.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Clustering groups observations by a chosen geometry without supervised task labels. The
resulting groups do not automatically correspond to meaningful business or network categories.
Scaling, feature representation, and density assumptions influence the result.

K-means alternates assigning points to nearby centroids and updating centroids. It requires k
and favors compact squared-Euclidean groups. Several restarts can reduce sensitivity to
initialization but do not guarantee a globally optimal partition.

DBSCAN groups density-connected observations and marks some points as noise with label -1.
eps and min_samples control the neighborhood policy. It does not require a cluster count,
but varying densities and high-dimensional distances can make one eps unsuitable.

Cluster numbers are arbitrary identifiers: cluster 0 is not intrinsically 'normal'. Silhouette
evaluates separation under a distance metric but is not a task-validity certificate. Synthetic
ground truth is available here only for an external demonstration metric, not for fitting.

## Mathematics or data contract

K-means minimizes the sum of squared distances to assigned centroids.

## Important limitation

The well-separated blobs deliberately suit K-means; evaluate a different geometry before generalizing.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Fit clusters and identify an explicit outlier

**Purpose:** Compare centroid grouping with density-based noise handling.

### Read the code in this order
1. Generate separated blobs and fit K-means without labels.
2. Measure permutation-invariant agreement only afterward.
3. Add a distant point for DBSCAN.
4. Check that noise is -1 rather than pretending it is a normal class.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
external synthetic agreement: 1.0
DBSCAN noise points: 1
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

X, hidden_labels = make_blobs(n_samples=120, centers=[[-3, -3], [3, 3]],
                             cluster_std=0.3, random_state=42)
groups = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)
agreement = adjusted_rand_score(hidden_labels, groups)
assert agreement > 0.9
with_outlier = np.vstack([X, [20, 20]])
density_groups = DBSCAN(eps=0.7, min_samples=4).fit_predict(with_outlier)
assert density_groups[-1] == -1
print("external synthetic agreement:", round(agreement, 3))
print("DBSCAN noise points:", int((density_groups == -1).sum()))

external synthetic agreement: 1.0
DBSCAN noise points: 1


### Interpret example 1

The result should demonstrate this purpose: **Compare centroid grouping with density-based noise handling.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Trace one K-means assignment/update step

**Purpose:** Understand the centroid arithmetic underneath the estimator.

### Read the code in this order
1. Create two well-separated groups and two starting centroids.
2. Assign each row to its nearest centroid.
3. Update each centroid and compare with the manually known means.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
assignments: [0, 0, 1, 1] updated centroids: [[0.0, 1.0], [10.0, 11.0]]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import numpy as np

X = np.array([[0., 0.], [0., 2.], [10., 10.], [10., 12.]])
centroids = np.array([[0., 0.], [10., 10.]])
distances = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
assignments = distances.argmin(axis=1)
updated = np.vstack([X[assignments == group].mean(axis=0) for group in range(2)])
assert assignments.tolist() == [0, 0, 1, 1]
np.testing.assert_allclose(updated, [[0., 1.], [10., 11.]])
print("assignments:", assignments.tolist(), "updated centroids:", updated.tolist())

assignments: [0, 0, 1, 1] updated centroids: [[0.0, 1.0], [10.0, 11.0]]


### Interpret example 2

The result should demonstrate this purpose: **Understand the centroid arithmetic underneath the estimator.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Try two moons and explain why K-means and density clustering behave differently.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Unsupervised learning` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: K-means minimizes the sum of squared distances to assigned centroids.
- Name the boundary of the demonstration: The well-separated blobs deliberately suit K-means; evaluate a different geometry before generalizing.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.